In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, precision_score, recall_score
from pathlib import Path

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, precision_score, recall_score
import pyarrow.parquet as pq
from pathlib import Path

print("=== AMAZON ML CHALLENGE 2026: MATCHING MODEL ===")

feature_cols = [
    'name_fuzz_ratio', 'name_token_sort', 
    'addr_fuzz_ratio', 'addr_token_sort',
    'exact_name_match', 's1_missing_addr', 
    'cand_missing_addr', 'both_missing_addr'
]

print("1. Loading features in memory-safe chunks...")
parquet_file = pq.ParquetFile("../output/training_features.parquet")
sampled_chunks = []

# Process 2 million rows at a time
for i, batch in enumerate(parquet_file.iter_batches(columns=feature_cols + ['label'], batch_size=2_000_000)):
    chunk = batch.to_pandas()
    
    # Downcast instantly to free RAM
    for col in chunk.columns:
        chunk[col] = chunk[col].astype(np.uint8)
        
    positives = chunk[chunk['label'] == 1]
    negatives = chunk[chunk['label'] == 0]
    
    # Keep 100% of matches, sample 25% of non-matches
    negatives_sampled = negatives.sample(frac=0.25, random_state=42)
    
    sampled_chunks.extend([positives, negatives_sampled])
    print(f"  Processed batch {i+1}...")

# Combine the chunks safely
df = pd.concat(sampled_chunks, ignore_index=True)
del sampled_chunks # Free memory

X = df[feature_cols]
y = df['label']

print(f"\nSuccessfully loaded and sampled down to {len(df):,} pairs.")
print(f"Positives: {y.sum():,} | Negatives: {len(y) - y.sum():,}")

# 2. Train / Validation Split
print("\n2. Splitting Train and Validation sets...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

del df # Free memory
del X, y 

# 3. Train XGBoost
print("\n3. Training XGBoost Classifier...")
# Calculate the new ratio based on our downsampled dataset
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=ratio,
    tree_method='hist',
    eval_metric='logloss',
    early_stopping_rounds=20,
    n_jobs=-1,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# 4. Threshold Optimization for F0.5
print("\n4. Optimizing Threshold for F0.5 Score...")
val_probs = model.predict_proba(X_val)[:, 1]

best_threshold = 0.50
best_f05 = 0.0
best_prec = 0.0
best_rec = 0.0

print(f"\n{'Threshold':<10} | {'F0.5 Score':<12} | {'Precision':<10} | {'Recall':<10}")
print("-" * 52)

for thresh in np.arange(0.50, 0.96, 0.05):
    preds = (val_probs >= thresh).astype(np.uint8)
    f05 = fbeta_score(y_val, preds, beta=0.5, zero_division=0)
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    
    print(f"{thresh:<10.2f} | {f05:<12.4f} | {prec:<10.4f} | {rec:<10.4f}")
    
    if f05 > best_f05:
        best_f05 = f05
        best_threshold = thresh
        best_prec = prec
        best_rec = rec

print(f"\n=== OPTIMAL THRESHOLD: {best_threshold:.2f} ===")
print(f"Peak F0.5:  {best_f05:.4f} (vs Baseline: 0.6010)")
print(f"Precision:  {best_prec:.4f}")
print(f"Recall:     {best_rec:.4f}")

# 5. Feature Importances
print("\n5. Feature Importances:")
for col, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f"  {col:<20}: {imp:.4f}")

# 6. Save Model
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)
model.save_model(output_dir / "xgb_matching_model.json")
with open(output_dir / "optimal_threshold.txt", "w") as f:
    f.write(str(best_threshold))
print(f"\nModel and threshold saved to {output_dir}")

=== AMAZON ML CHALLENGE 2026: MATCHING MODEL ===
1. Loading features in memory-safe chunks...
  Processed batch 1...
  Processed batch 2...
  Processed batch 3...
  Processed batch 4...
  Processed batch 5...
  Processed batch 6...
  Processed batch 7...
  Processed batch 8...
  Processed batch 9...
  Processed batch 10...
  Processed batch 11...
  Processed batch 12...
  Processed batch 13...
  Processed batch 14...
  Processed batch 15...
  Processed batch 16...
  Processed batch 17...
  Processed batch 18...
  Processed batch 19...
  Processed batch 20...
  Processed batch 21...
  Processed batch 22...
  Processed batch 23...
  Processed batch 24...
  Processed batch 25...
  Processed batch 26...
  Processed batch 27...
  Processed batch 28...
  Processed batch 29...
  Processed batch 30...
  Processed batch 31...
  Processed batch 32...
  Processed batch 33...

Successfully loaded and sampled down to 20,672,527 pairs.
Positives: 5,565,452 | Negatives: 15,107,075

2. Splitting Train

In [3]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, fbeta_score, precision_score, recall_score
import pyarrow.parquet as pq

print("1. Loading saved model and threshold...")
model = xgb.XGBClassifier()
model.load_model("../output/xgb_matching_model.json")

with open("../output/optimal_threshold.txt", "r") as f:
    thresh = float(f.read())

print(f"Using Optimal Threshold: {thresh}")

print("\n2. Loading a sample chunk to test...")
# Load 1 million rows just to calculate the final metrics
parquet_file = pq.ParquetFile("../output/training_features.parquet")
df = next(parquet_file.iter_batches(batch_size=1_000_000)).to_pandas()

feature_cols = [
    'name_fuzz_ratio', 'name_token_sort', 
    'addr_fuzz_ratio', 'addr_token_sort',
    'exact_name_match', 's1_missing_addr', 
    'cand_missing_addr', 'both_missing_addr'
]

X = df[feature_cols]
y = df['label']

print("3. Calculating predictions...")
probs = model.predict_proba(X)[:, 1]
preds = (probs >= thresh).astype(int)

acc = accuracy_score(y, preds)
f05 = fbeta_score(y, preds, beta=0.5, zero_division=0)
prec = precision_score(y, preds, zero_division=0)
rec = recall_score(y, preds, zero_division=0)

print("\n=== FINAL MODEL PERFORMANCE ===")
print(f"Accuracy:   {acc*100:.2f}%")
print(f"F0.5 Score: {f05:.4f}  <-- Competition Metric")
print(f"Precision:  {prec:.4f}")
print(f"Recall:     {rec:.4f}")

1. Loading saved model and threshold...
Using Optimal Threshold: 0.8500000000000003

2. Loading a sample chunk to test...
3. Calculating predictions...

=== FINAL MODEL PERFORMANCE ===
Accuracy:   95.98%
F0.5 Score: 0.7621  <-- Competition Metric
Precision:  0.7619
Recall:     0.7629


<h2> Performing HyperParameter tuning



In [4]:
!pip install optuna


   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ---------------------------------------- 4/4 [optuna]




[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
